# Progetto — Fondamenti di Analisi dei Dati

**Gruppo:** Nome e Cognome · Nome e Cognome · Nome e Cognome  
**Dataset assegnato:** *(nome)*

Questo notebook è vuoto di proposito: la struttura dell'analisi è parte del lavoro.
I due brief dicono che cosa deve contenere — [Parte 1](../docs/parte_1.md) e [Parte 2](../docs/parte_2.md) — e la Parte 2 prosegue qui sotto, dove finisce la Parte 1.

I dati si leggono da `../data/`, che resta vuota nel repository: `data/README.md` dice come procurarseli.


# Parte 1 — Descrivere, inferire, spiegare

## 0. Setup
Librerie e caricamento dei dati dalla cartella `Dataset` su Google Drive.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)   # mostra tutte le colonne
pd.set_option("display.max_rows", 100)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA = "/content/drive/MyDrive/Dataset/"     # attenzione alla / finale
!ls "{DATA}"

Mounted at /content/drive
diabetic_data.csv


In [3]:
df = pd.read_csv(DATA + "diabetic_data.csv",
                 na_values="?",            # nel file i mancanti sono scritti "?"
                 keep_default_na=False,    # "None" in A1Cresult/max_glu_serum = esame non fatto, non un mancante
                 low_memory=False)

## 1. Comprensione del dataset

### 1.1 Caricamento e prima ispezione

In [4]:
print("Righe:", df.shape[0])
print("Colonne:", df.shape[1])
print(df.columns.tolist())

Righe: 101766
Colonne: 50
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [5]:
print(df.dtypes.value_counts(), "\n")
df.dtypes

object    37
int64     13
Name: count, dtype: int64 



,0
encounter_id,int64
patient_nbr,int64
race,object
gender,object
age,object
weight,object
admission_type_id,int64
discharge_disposition_id,int64
admission_source_id,int64
time_in_hospital,int64


In [6]:
df.sample(10, random_state=0)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
43462,134140482,88421031,Caucasian,Male,[70-80),NaN,3,1,1,3,NaN,InternalMedicine,44,2,12,1,0,0,574,576,250,7,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,<30
33680,106194426,23205420,AfricanAmerican,Female,[80-90),NaN,1,1,7,2,NaN,Nephrology,42,2,18,0,2,0,535,403,531,8,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO
32406,103343040,23241960,Caucasian,Female,[70-80),NaN,2,1,1,4,NaN,Orthopedics-Reconstructive,41,1,35,0,0,0,812,428,428,7,None,None,Steady,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,>30
89036,287173320,96539697,Caucasian,Male,[70-80),NaN,1,6,7,2,UN,NaN,53,0,20,7,0,2,250.82,511,276,9,None,>8,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,<30
81398,252179430,32240799,Caucasian,Female,[70-80),NaN,1,13,7,6,MC,NaN,59,0,13,0,0,0,434,496,585,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Down,No,No,No,No,No,Ch,Yes,>30
2476,16187658,5535711,AfricanAmerican,Female,[50-60),NaN,1,1,7,4,NaN,Gastroenterology,59,3,11,0,0,1,285,276,535,9,None,Norm,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
16490,61844922,10238310,AfricanAmerican,Male,[10-20),NaN,1,1,7,3,NaN,Anesthesiology-Pediatric,60,1,9,0,0,1,250.13,287,276,5,None,Norm,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
69861,199323216,86413257,Caucasian,Female,[70-80),NaN,1,1,7,6,HM,NaN,48,6,13,0,0,0,415,414,401,7,None,None,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,>30
32445,103433490,14346522,Caucasian,Female,[70-80),NaN,2,1,1,2,MC,Gastroenterology,37,1,9,0,0,0,557,414,V10,5,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,NO
38516,119742684,48761964,Caucasian,Male,[60-70),NaN,2,1,7,4,MC,InternalMedicine,73,2,13,3,0,0,574,574,786,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
